# 06 — Volatility Forecast & Evaluation

This notebook generates out-of-sample rolling volatility forecasts and
evaluates them using RMSE, MAE, and the QLIKE loss function.

**Pipeline step 7 of 7.**

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.data_loader import load_processed_data
from src.preprocessing import prepare_returns_for_garch
from src.model_builder import ModelSpec
from src.forecast import rolling_forecast, evaluate_forecast, save_forecast_results
from pathlib import Path

%matplotlib inline

## 6.1 Load Data

In [ ]:
df = load_processed_data()
returns_pct = prepare_returns_for_garch(df["log_return"].dropna())
print(f"Total observations: {len(returns_pct)}")

## 6.2 Rolling Forecast — GARCH(1,1) Normal

We use 80% of the data for training and roll forward one step at a time.

In [ ]:
spec_normal = ModelSpec("GARCH", p=1, q=1, distribution="normal")
forecast_df_normal = rolling_forecast(
    returns_pct,
    model_spec=spec_normal,
    train_size=0.8,
    refit_frequency=5,  # Refit every 5 steps for speed
)
forecast_df_normal.head()

## 6.3 Evaluate Forecast

In [ ]:
metrics_normal = evaluate_forecast(forecast_df_normal)
print("GARCH(1,1) Normal Distribution Metrics:")
for k, v in metrics_normal.items():
    print(f"  {k}: {v:.6f}")

## 6.4 Plot Forecast vs Actual Squared Returns

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(forecast_df_normal.index, forecast_df_normal["actual_sq_return"],
        linewidth=0.6, color="steelblue", alpha=0.7, label="Squared Returns (Actual)")
ax.plot(forecast_df_normal.index, forecast_df_normal["forecast_variance"],
        linewidth=1.0, color="coral", label="GARCH(1,1) Forecast Variance")
ax.set_title("Rolling Volatility Forecast vs Actual (GARCH(1,1) Normal)")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.savefig("../results/figures/rolling_forecast_normal.png", dpi=150)
plt.show()

## 6.5 Save Results

In [ ]:
save_forecast_results(
    forecast_df_normal,
    metrics_normal,
    output_dir=Path("../results/tables"),
    model_name="GARCH11_normal",
)
print("Results saved to results/tables/")

## 6.6 Summary Table

In [ ]:
summary = pd.DataFrame([{"Model": "GARCH(1,1) Normal", **metrics_normal}])
print(summary.to_string(index=False))